## Creación de Base clean

In [105]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import holidays

# Configura Pandas para mostrar floats sin notación científica
pd.set_option('display.float_format', lambda x: '%.0f' % x)

In [106]:
PATH_ORDERS  = "Orders.csv"
PATH_DETAILS = "details_clean.csv"
PATH_RESULTS = "Resultados_sin_hash.csv"

df_orders    = pd.read_csv(PATH_ORDERS, dtype={'id_pedido': str})
df_details   = pd.read_csv(PATH_DETAILS, dtype={'id_pedido': str})
df_resultados = pd.read_csv(PATH_RESULTS, dtype={'id_pedido': str})

In [107]:
# Forzar a los tres DataFrames a ser flotantes numéricos
df_orders['id_pedido'] = df_orders['id_pedido'].astype(float)
df_details['id_pedido'] = df_details['id_pedido'].astype(float)
df_resultados['id_pedido'] = df_resultados['id_pedido'].astype(float)

In [108]:
df_orders['customer_id'] = df_orders['customer_id'].astype(float)

In [109]:
df_orders['fecha_pedido'] = pd.to_datetime(df_orders['fecha_pedido'], errors='coerce')

/var/folders/gc/wcr1rzm57ls2jc6mkw7j8xr80000gn/T/ipykernel_56824/1516480952.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_orders['fecha_pedido'] = pd.to_datetime(df_orders['fecha_pedido'], errors='coerce')


In [110]:
df_orders = df_orders[df_orders['pais'] == 'México']

In [111]:
# Mantener unicamente el primer registro de cada id_pedido en df_orders
df_orders = df_orders.drop_duplicates(subset='id_pedido', keep='first')

In [112]:
# Dropear fecha_entrega
df_orders = df_orders.drop(columns=['fecha_entrega'], errors='ignore')

In [113]:
# 1. Definimos el rango completo de días posibles entre 2022 y 2025
rango_fechas = pd.date_range(start='2022-01-01', end='2025-12-31', freq='h')

# 2. Creamos la columna 'fecha_pedido' eligiendo fechas al azar de ese rango
#    usamos len(df_orders) para generar una fecha para cada fila de tu DataFrame
df_orders['fecha_pedido'] = np.random.choice(rango_fechas, size=len(df_orders))

In [114]:
df_orders.fecha_pedido.value_counts()

fecha_pedido
2024-10-26 03:00:00    7
2022-09-05 19:00:00    6
2024-09-25 21:00:00    6
2024-03-20 02:00:00    6
2025-07-18 11:00:00    5
                      ..
2025-07-05 21:00:00    1
2022-09-01 03:00:00    1
2023-07-25 06:00:00    1
2025-12-27 23:00:00    1
2023-02-10 11:00:00    1
Name: count, Length: 18945, dtype: int64

In [115]:
for df in [df_orders, df_details, df_resultados]:
    df['id_pedido'] = pd.to_numeric(df['id_pedido'], errors='coerce')

In [116]:
# Merge correcto para modelo:
# 1) Orders + Details por id_pedido: baja la informacion del pedido a cada linea.
# 2) Resultados por id_linea: marca solo la linea exacta que fue sustituida.
#
# No usamos df_resultados completo en el merge porque trae columnas repetidas
# y porque unirlo por id_pedido multiplica filas artificialmente.
cols_cambio = [
    'id_linea',
    'sku_solicitado_cambio',
    'nombre_sku_solicitado_cambio',
]

df_resultados_cambios = df_resultados[cols_cambio].drop_duplicates(subset='id_linea')

df = (
    df_orders
    .merge(df_details, on='id_pedido', how='left')
    .merge(df_resultados_cambios, on='id_linea', how='left')
)

# Para el modelo, cada registro debe ser una linea unica de pedido.
df = df.dropna(subset=['id_linea']).copy()
df = df.drop_duplicates(subset='id_linea', keep='first')

# Target: una linea fue sustituida si existe en Resultados.
df['fue_sustituida'] = df['sku_solicitado_cambio'].notna().astype(int)

# Guardamos version completa para auditoria, con el SKU de cambio visible.
df.to_csv('merged_con_cambio.csv', index=False)

# Guardamos version para modelo sin columnas de fuga.
df_modelo = df.drop(columns=[
    'sku_solicitado_cambio',
    'nombre_sku_solicitado_cambio',
])

df_modelo.to_csv('merged.csv', index=False)

print('merged_con_cambio.csv:', df.shape)
print('merged.csv para modelo:', df_modelo.shape)
print('id_linea duplicados:', df_modelo['id_linea'].duplicated().sum())
print('filas completas duplicadas:', df_modelo.duplicated().sum())
print('columnas _x/_y:', [col for col in df_modelo.columns if col.endswith(('_x', '_y'))])
print(df_modelo['fue_sustituida'].value_counts())



merged_con_cambio.csv: (935557, 19)
merged.csv para modelo: (935557, 17)
id_linea duplicados: 0
filas completas duplicadas: 0
columnas _x/_y: []
fue_sustituida
0    932232
1      3325
Name: count, dtype: int64
